In [2]:
library(DEqMS)
library(patchwork)
library(tidyverse)

source("../../evaluation_utils/evaluation/DE_analysis.R")
source("../../evaluation_utils/plots/DE_plots.R")
source("../../evaluation_utils/filtering/filtering_normalization.R")

source("../../evaluation_utils/plots/eda_plots.R")

Loading required package: ggplot2

Loading required package: matrixStats

Loading required package: limma

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ lubridate 1.9.4     ✔ tibble    3.2.1
✔ purrr     1.0.2     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::count()  masks matrixStats::count()
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose



Attaching pa

# check datasets

In [3]:
study_list = c('PDC000127', 'PXD030344', 'PXD042844')
path_to_reports = paste0('/home/yuliya/repos/cosybio/FedProt/data/ccRCC_studies/data/')


central_batch_info = NULL
central_intensities = NULL

for (name in study_list) {
  batch_info = read_tsv(paste0(path_to_reports, name, '/metadata.csv'), show_col_types = FALSE)
  intensities = read_csv(paste0(path_to_reports, name, '/report_filtered.csv'), show_col_types = FALSE)
  
  print(paste0('Processing ', name))
  print(paste0('Number of samples: ', ncol(batch_info), "; Number of proteins: ", nrow(intensities)))

  if(is.null(central_intensities)){
    central_intensities = intensities
    central_batch_info = batch_info
  } else {
    central_intensities = full_join(central_intensities, intensities, by = 'Gene')
    central_batch_info = rbind(central_batch_info, batch_info)
  }
}
central_batch_info <- central_batch_info %>%
    mutate(Dataset = as.factor(Dataset), Condition = as.factor(Condition))

cat('\tNumber of proteins: ', nrow(central_intensities), '\n')
cat('\tNumber of samples: ', ncol(central_intensities)-1, '\n')

central_intensities <- central_intensities %>%
  column_to_rownames('Gene')
central_intensities <- central_intensities[, central_batch_info$Sample]

cat('\tNumber of proteins: ', nrow(central_intensities), '\n')

# order columns
central_intensities <- central_intensities[, central_batch_info$Sample]

[1] "Processing PDC000127"
[1] "Number of samples: 3; Number of proteins: 9964"
[1] "Processing PXD030344"
[1] "Number of samples: 3; Number of proteins: 12548"
[1] "Processing PXD042844"
[1] "Number of samples: 3; Number of proteins: 7655"
	Number of proteins:  13826 
	Number of samples:  887 
	Number of proteins:  13826 


In [8]:
central_batch_info %>%
  group_by(Dataset, Condition) %>%
  summarise(Count = n())

`summarise()` has grouped output by 'Dataset'. You can override using the
`.groups` argument.


Dataset,Condition,Count
<fct>,<fct>,<int>
PDC000127,Normal,84
PDC000127,Tumor,110
PXD030344,Normal,232
PXD030344,Tumor,232
PXD042844,Normal,114
PXD042844,Tumor,115


In [4]:
central_intensities <- filter_na_proteins(central_intensities, central_batch_info, "Sample")
central_intensities <- filter_by_condition(
    central_intensities, central_batch_info,
    'Sample', c('Tumor', 'Normal'), 'Condition'
    )

Filtering out features that have NAs in all columns
	Before filtering: 13826 887 
	After filtering: 13826 887 
Filtering by condition - min_f not-NA per condition
	Before filtering: 13826 887 
	After filtering: 9605 887 


## BEC for QC

In [ ]:
# batch effects correction
design <- model.matrix(~ Condition, data = central_batch_info)
pg_corrected <- removeBatchEffect(
  central_intensities,
  batch = central_batch_info$Dataset,
  design = design
)

# write data to file
write.csv(pg_corrected, file = "/home/yuliya/repos/cosybio/FedProt/data/ccRCC_studies/data/ccRCC_data_afterBEC.csv")
# write metadata to file
write.csv(central_batch_info, file = "/home/yuliya/repos/cosybio/FedProt/data/ccRCC_studies/data/ccRCC_metadata.csv")



Warning message:
“Partial NA coefficients for 3342 probe(s)”
Warning message:
“`aes_string()` was deprecated in ggplot2 3.0.0.
ℹ Please use tidy evaluation idioms with `aes()`.
ℹ See also `vignette("ggplot2-in-packages")` for more information.”


In [6]:
plot_pca_before <- pca_plot(
  central_intensities, central_batch_info, 
  title=paste0("ccRCC dataset"), 
  quantitative_col_name='Sample', 
  col_col='Condition', shape_col='Dataset',
  size_point=1.5
)

ggsave(file = "plots/PCA_plot.svg", plot = plot_pca_before, width = 5, height = 5)


plot_pca_after <- pca_plot(
  pg_corrected, central_batch_info, 
  title=paste0("ccRCC dataset after correction (removeBatchEffect)"), 
  quantitative_col_name='Sample', 
  col_col='Condition', shape_col='Dataset',
  size_point=1.5
)

ggsave(file = "plots/PCA_plot_afterBEC.svg", plot = plot_pca_after, width = 5, height = 5)


# Central analysis

In [ ]:
# run DE analysis
design <- make_design(central_batch_info, 'Condition', 'Dataset')
contrasts <- makeContrasts(Normal - Tumor, levels = colnames(design))
de_results <- run_DE(central_intensities, NULL, design, contrasts)
de_results <- de_results %>% rownames_to_column('Gene')

write.table(
    de_results, 
    file = paste0('results/central_res.tsv'), 
    sep = "\t", quote = FALSE, row.names = FALSE)

# plot volcano plot
plot_result <- volcano_plot(
    de_results, paste("ccRCC data,", "central", ",Control/Tumor"),
    pval_threshold = 0.05, logfc_threshold = 0.5,
    show_names = FALSE
)
ggsave(
    file = paste0('plots/central_volcano_plot.svg'), 
    plot = plot_result, width = 8, height = 5)



Warning message:
“Partial NA coefficients for 3342 probe(s)”


Count information is not available
Using P.Value and adj.P.Val as sca.P.Value and sca.adj.pval


# Meta-analyses

In [26]:
options(warn=-1)

# empty plot
x <- ggplot() + theme_minimal()

plots_list = list()

for (name in study_list) {
  output_path = paste0('Meta_DE/')

  batch_info = read_tsv(
    paste0(path_to_reports, name, '/metadata.csv'), show_col_types = FALSE
  )
  intensities = read_csv(
    paste0(path_to_reports, name, '/report_filtered.csv'), show_col_types = FALSE
  )
  
  print(paste0('Processing ', name))
  print(paste0('Number of samples: ', ncol(batch_info), "; Number of proteins: ", nrow(intensities)))

  # create design matrix for FedProt
  rownames(batch_info) <- batch_info$Sample
  dummy_df <- model.matrix(~Condition - 1, batch_info)
  colnames(dummy_df) <- gsub("Condition", "", colnames(dummy_df))
  design <- batch_info %>% 
    select(-c("Condition")) %>% 
    cbind(dummy_df)
  write_tsv(
    design %>% rownames_to_column(), 
    paste0(path_to_reports, name, "/design.tsv")
  )

  intensities <- intensities %>%
    column_to_rownames('Gene')
  # filter
  intensities <- filter_by_condition(
    intensities, batch_info,
    'Sample', c('Tumor', 'Normal'), 'Condition'
  )
  intensities <- filter_na_proteins(
    intensities, batch_info, "Sample"
  )
  
  # order
  intensities <- intensities[, batch_info$Sample]

  # run DE
  design <- make_design(batch_info, 'Condition')
  contrasts <- makeContrasts(Normal - Tumor, levels = colnames(design))
  de_results <- run_DE(intensities, NULL, design, contrasts)
  de_results <- de_results %>% rownames_to_column('Gene')
  write.table(de_results, file = paste0(output_path, name, '_res.tsv'), sep = "\t", quote = FALSE, row.names = FALSE)

  # plot volcano plots
  if(name == 'PXD042844'){
    plot_separate <- volcano_plot(
      de_results, paste0("ccRCC data, ", name, ", Normal/Tumor"),
      pval_threshold = 0.05, logfc_threshold = 0.5,
      show_names = FALSE
    )
  } else {
    plot_separate <- volcano_plot(
      de_results, paste0("ccRCC data, ", name, ", Normal/Tumor"),
      pval_threshold = 0.05, logfc_threshold = 0.5,
      show_names = FALSE, show_legend = FALSE
    )
  }
  plots_list[[name]] <- plot_separate
}

layout <- (plots_list[['PDC000127']] | plots_list[['PXD030344']] | plots_list[['PXD042844']])
# save plot
ggsave(file = paste0(output_path, "volcano_plots.svg"), plot = layout, width = 18, height = 7)



[1] "Processing PDC000127"
[1] "Number of samples: 3; Number of proteins: 9964"
Filtering by condition - min_f not-NA per condition
	Before filtering: 9964 194 
	After filtering: 9896 194 
Filtering out features that have NAs in all columns
	Before filtering: 9896 194 
	After filtering: 9896 194 
Count information is not available
Using P.Value and adj.P.Val as sca.P.Value and sca.adj.pval
[1] "Processing PXD030344"
[1] "Number of samples: 3; Number of proteins: 12548"
Filtering by condition - min_f not-NA per condition
	Before filtering: 12548 464 
	After filtering: 9320 464 
Filtering out features that have NAs in all columns
	Before filtering: 9320 464 
	After filtering: 9320 464 
Count information is not available
Using P.Value and adj.P.Val as sca.P.Value and sca.adj.pval
[1] "Processing PXD042844"
[1] "Number of samples: 3; Number of proteins: 7655"
Filtering by condition - min_f not-NA per condition
	Before filtering: 7655 229 
	After filtering: 6064 229 
Filtering out features 

# Meta run

In [27]:
system(paste0("cd /home/yuliya/repos/cosybio/FedProt/evaluation/ccRCC_studies/Meta_DE/"))

system(paste0("Rscript /home/yuliya/repos/cosybio/FedProt/evaluation_utils/meta_code/run_MetaDE.R /home/yuliya/repos/cosybio/FedProt/evaluation/ccRCC_studies/Meta_DE/ PDC000127 PXD030344 PXD042844"))
system(paste0("Rscript /home/yuliya/repos/cosybio/FedProt/evaluation_utils/meta_code/run_MetaVolcanoR.R /home/yuliya/repos/cosybio/FedProt/evaluation/ccRCC_studies/Meta_DE/ PDC000127 PXD030344 PXD042844"))
system(paste0("Rscript /home/yuliya/repos/cosybio/FedProt/evaluation_utils/meta_code/run_RankProd.R /home/yuliya/repos/cosybio/FedProt/evaluation/ccRCC_studies/Meta_DE/ PDC000127 PXD030344 PXD042844"))

# Copy the resulting files to the desired directory
system(paste0("cp /home/yuliya/repos/cosybio/FedProt/evaluation/ccRCC_studies/Meta_DE/MA_* /home/yuliya/repos/cosybio/FedProt/evaluation/ccRCC_studies/results/"))


# FedProt run

In [28]:
# read each intensities file and conver it to tsv
for (name in study_list) {
  intensities = read_csv(paste0(path_to_reports, name, '/report_filtered.csv'), show_col_types = FALSE)
  write_tsv(intensities, paste0(path_to_reports, name, '/report_filtered.tsv'))
}

In [29]:
for(lab in c('PDC000127', 'PXD030344', 'PXD042844')){
  system(
    paste0(
      "cp /home/yuliya/repos/cosybio/FedProt/data/ccRCC_studies/data/", lab, "/design.tsv /home/yuliya/repos/cosybio/FedProt/data/ccRCC_studies/", lab, "/design.tsv"
    )
  )
  system(
    paste0(
      "cp /home/yuliya/repos/cosybio/FedProt/data/ccRCC_studies/data/", lab, "/report_filtered.tsv /home/yuliya/repos/cosybio/FedProt/data/ccRCC_studies/", lab, "/report_filtered.tsv"
    )
  )
}

In [30]:
system(
  "python /home/yuliya/repos/cosybio/FedProt/evaluation_utils/fedprot_prototype/fedprot_script.py /home/yuliya/repos/cosybio/FedProt/data ccRCC_studies PDC000127,PXD030344,PXD042844 /home/yuliya/repos/cosybio/FedProt/evaluation/"
)

# Session info

In [31]:
# all versions of all used packages print
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 24.04.1 LTS

Matrix products: default
BLAS/LAPACK: /home/yuliya/miniforge3/envs/FedProt/lib/libopenblasp-r0.3.28.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] grid      stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] gridExtra_2.3     foreach_1.5.2     data.table_1.16.4 ggrepel_0.9.6    
 [5] lubridate_1.9.4   forcats_1.0.0     stringr_1.5.1     dplyr_1.1.4      
 [9] purrr_1.0.2       readr_2.1.5       tidyr_1.3.1   